# Experimento: ¿mejora el modelo con calibración o con variables nuevas?

Comparamos tres ideas contra el Elo actual, **con los mismos partidos y la misma regla de decisión**:

- **A. Calibración por temperatura**: corregir la confianza del Elo sin agregar información.
- **B. Variables nuevas** (regresión logística):
  - *Estilo de equipo*: agresividad temprana, duración de las partidas, prioridad de objetivos, e interacciones entre estilos ("piedra, papel o tijera").
  - *Forma por rol*: cómo viene jugando cada jugador comparado con el promedio de su posición.

Datos: Oracle's Elixir (stats por jugador y por partida, incluidas las diferencias al minuto 15). Se bajan con `python -m pipeline.fetch_oracles`.

**Regla de decisión (fijada antes de mirar los resultados):** una variante *mejora* si

1. el intervalo de confianza del 95% de su mejora de Brier en **2025 (todas las partidas)** no incluye el 0, y
2. no empeora en las partidas **entre regiones** de 2025 (las que más se parecen a Worlds).

2026 (hasta antes de Worlds) queda como segunda confirmación.

¿Por qué fijar la regla antes? Si probás muchas variantes y después elegís el criterio que favorece a la que te gusta, casi siempre "encontrás" una mejora aunque no exista. Es el error de los *caminos que se bifurcan* (garden of forking paths).

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # el notebook corre desde notebooks/

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from models.backtest import cross_region_records, tune
from models.calibration import fit_temperature, scale_probability
import models.calibration as calibration_module
from models.elo import run_elo, series_probability, team_regions
from models.features import ROLE_FEATURES, STYLE_FEATURES, build_dataset, fit_norms
from models.metrics import accuracy, bootstrap_brier_diff, brier_score, log_loss
from models.oracles import load_season

OE_DIR = Path.cwd().parent / "data" / "raw" / "oracles_elixir"
YEARS = (2024, 2025, 2026)

## 1. Los datos y lo que les falta

Antes de modelar, siempre: ¿cuántos datos hay y qué falta? Un dato faltante que no se ve puede arruinar una conclusión.

In [2]:
raw = {y: pd.read_csv(OE_DIR / f"{y}.csv", low_memory=False) for y in YEARS}
main_leagues = ["LCK", "LPL", "LEC", "LCS", "LTA N", "LTA S", "CBLOL", "LCP", "PCS", "VCS", "LJL", "MSI", "WLDs", "FST", "EWC"]
missing = []
for y, df in raw.items():
    teams = df[(df.position == "team") & df.league.isin(main_leagues)]
    g = teams.groupby("league").agg(partidas=("gameid", "nunique"),
                                    sin_oro_al_15=("golddiffat15", lambda s: round(s.isna().mean(), 2)))
    g["año"] = y
    missing.append(g.reset_index())
pd.concat(missing).pivot(index="league", columns="año", values="sin_oro_al_15").fillna("-")

año,2024,2025,2026
league,,,
CBLOL,0.0,-,0.0
EWC,0.0,0.0,0.0
FST,-,0.0,0.0
LCK,0.0,0.0,0.0
LCP,-,0.0,0.0
LCS,0.0,-,0.0
LEC,0.0,0.0,0.0
LJL,0.0,0.0,0.0
LPL,1.0,1.0,0.2


**Hallazgo importante:** la LPL de 2024 y 2025 (y el MSI 2024) **no tienen** stats al minuto 15: 100% faltante. En 2026 la LPL ya trae el 80%.

Consecuencias:
- Para los jugadores chinos de 2024/2025, la "forma" se calcula solo con KP, muertes, % de daño y eficiencia, sin la fase de líneas.
- Un dato faltante cuenta como "promedio" (no suma ni resta). Es la opción prudente, pero hace que las variables ayuden **menos** con equipos chinos. Justo una de las regiones fuertes de Worlds.

In [3]:
t0 = time.time()
games = {y: load_season(OE_DIR / f"{y}.csv") for y in YEARS}
regions = {y: team_regions(games[y]) for y in YEARS}
{y: len(g) for y, g in games.items()}

{2024: 3087, 2025: 3496, 2026: 3367}

## 2. El Elo base, ajustado solo con 2024

Para que la comparación sea justa, **ningún** parámetro puede haber visto 2025. El Elo de producción se ajustó con 2024 + 2025, así que acá se reajusta K y los ajustes regionales solo con 2024 (con el mismo `tune` del backtest original) sobre estas partidas.

In [4]:
params, score_2024 = tune([(games[2024], regions[2024])])
print(params, round(score_2024, 4))
runs = {y: run_elo(games[y], params, regions[y]) for y in YEARS}

Params(k=15.0, region_offsets={'KR': 350.0, 'CN': 350.0, 'EU': 200.0, 'NA': 200.0}) 0.2181


## 3. Las variables nuevas

`models/features.py` recorre las partidas en orden. Para cada una, **primero** arma las variables con lo que pasó antes y **después** aprende de esa partida. Así no hay *fuga de información* (leakage): si una variable usara el resultado de la partida que predice, el modelo parecería genial en el backtest y sería inútil en la realidad. Hay un test que lo verifica.

Cómo se construye cada variable:

- **z-score:** cada métrica se expresa en "desvíos respecto del promedio". Para los jugadores, el promedio es **el de su rol**: un support con 2 kills es normal, un mid con 2 kills no. Los promedios y desvíos salen de 2024 y se usan igual en todos los años.
- **Promedio móvil exponencial (EWMA)** con media vida de 10 partidas: la última partida pesa el doble que la de hace 10. Combina la forma reciente con el nivel de la temporada.
  - Todos arrancan en 0, el promedio. Un jugador nuevo queda cerca del promedio hasta acumular partidas: es una forma simple de no creerle a muestras chicas (*shrinkage*).
- **Estilo de equipo:**
  - agresividad = kills al minuto 15 + primera sangre;
  - duración = minutos de partida;
  - objetivos = primer dragón, primera torre, primer heraldo.
- **Forma por rol:**
  - diferencia de oro, experiencia y farmeo al 15 contra el rival directo;
  - KP;
  - muertes (con signo invertido);
  - % del daño del equipo;
  - eficiencia (% de daño / % de oro): separa "jugar mal" de "jugar con pocos recursos".
- **Antisimetría:** todas las variables son "A menos B". Si se dan vuelta los equipos, cambian de signo y la probabilidad pasa a ser 1 − p, como tiene que ser.
  - Las interacciones se arman como `agresión_A × duración_B − agresión_B × duración_A`.
  - El producto de dos diferencias **no** sirve: (−x)(−y) = xy, no cambia de signo.

In [5]:
norms = fit_norms(games[2024])
data = {y: build_dataset(runs[y].records, norms) for y in YEARS}
df25 = pd.DataFrame(data[2025])
df25[list(STYLE_FEATURES + ROLE_FEATURES)].describe().round(2)

,d_aggr,d_dur,d_obj,x_aggr_dur,x_obj_dur,d_top,d_jng,d_mid,d_bot,d_sup
count,3496.00,3496.00,3496.00,3496.00,3496.00,3496.00,3496.00,3496.00,3496.00,3496.00
mean,-0.02,-0.00,-0.02,-0.00,0.00,-0.01,-0.02,-0.01,-0.01,-0.01
std,0.26,0.25,0.22,0.08,0.06,0.19,0.23,0.23,0.16,0.14
min,-1.31,-0.91,-0.94,-0.61,-0.37,-0.70,-0.80,-0.87,-0.65,-0.53
25%,-0.18,-0.16,-0.16,-0.03,-0.02,-0.12,-0.16,-0.15,-0.12,-0.09
50%,-0.01,0.00,-0.02,0.00,0.00,-0.01,-0.01,-0.01,-0.01,-0.00
75%,0.14,0.15,0.12,0.02,0.02,0.11,0.13,0.13,0.09,0.08
max,1.34,0.96,0.77,0.71,0.41,0.80,1.02,0.86,0.57,0.52


## 4. Las variantes

| Variante | Qué es |
|---|---|
| **Base** | Elo (ajustado con 2024) |
| **A** | Elo × temperatura T. T se elige en 2024, de dos formas: con todas las partidas y solo con las de entre regiones |
| **B0** | Regresión logística solo con el Elo y el lado. Sirve de control: muestra cuánto cambia por re-pesar el Elo, sin variables nuevas |
| **B-estilo / B-roles / B** | Regresión logística con Elo + lado + estilo / roles / todo |

**Regresión logística:** el log-odds de ganar se modela como una suma ponderada,
`log(p / (1 − p)) = b₀ + b₁·elo + b₂·d_top + …`, y la regresión aprende los pesos.

- Si la única variable es el Elo, la regresión **es** una calibración: aprende cuánto creerle. Por eso B0 es el control justo.
- **Regularización L2 (C):** castiga los pesos grandes para no sobreajustar. C se elige cortando 2024 en el 70% más viejo (entrenar) y el 30% más nuevo (probar), respetando el orden del tiempo.

In [6]:
FEATURES = {
    "B0": ("elo_logit",),
    "B-estilo": ("elo_logit",) + STYLE_FEATURES,
    "B-roles": ("elo_logit",) + ROLE_FEATURES,
    "B": ("elo_logit",) + STYLE_FEATURES + ROLE_FEATURES,
}

def matrix(rows, cols):
    return np.array([[r[c] for c in cols] for r in rows])

def outcomes(rows):
    return np.array([r["blue_won"] for r in rows])

train = data[2024]
cut = int(len(train) * 0.7)
logistic = {}
for name, cols in FEATURES.items():
    scores = {}
    for C in (0.001, 0.01, 0.1, 1, 10):
        scaler = StandardScaler().fit(matrix(train[:cut], cols))
        m = LogisticRegression(C=C).fit(scaler.transform(matrix(train[:cut], cols)), outcomes(train[:cut]))
        p = m.predict_proba(scaler.transform(matrix(train[cut:], cols)))[:, 1]
        scores[C] = brier_score(list(p), list(outcomes(train[cut:])))
    best_c = min(scores, key=scores.get)
    scaler = StandardScaler().fit(matrix(train, cols))
    logistic[name] = (scaler, LogisticRegression(C=best_c).fit(scaler.transform(matrix(train, cols)), outcomes(train)), cols)
    print(f"{name:9} C={best_c}")

# Temperatura con la grilla ampliada (0.30–2.00): el óptimo con todas las partidas queda > 1.5
calibration_module.T_GRID = tuple(round(0.3 + i * 0.01, 2) for i in range(171))
cross_2024 = cross_region_records(train, regions[2024])
T_all = fit_temperature([r["p_elo"] for r in train], [r["blue_won"] for r in train])
T_cross = fit_temperature([r["p_elo"] for r in cross_2024], [r["blue_won"] for r in cross_2024])
print("T (todas 2024) =", T_all, "| T (entre regiones 2024) =", T_cross)

B0        C=0.01
B-estilo  C=0.01
B-roles   C=0.01
B         C=0.01


T (todas 2024) = 1.83 | T (entre regiones 2024) = 1.15


**Primera sorpresa:** las dos T salen **mayores que 1**.

- T > 1 significa que el Elo ajustado con 2024 se queda **corto** de confianza: el modelo tendría que estar más seguro, no menos.
- Dentro de cada liga se nota mucho (T = 1.83). Es lógico: K y los ajustes regionales se eligieron pensando en los cruces entre regiones, no en las ligas.
- La sobreconfianza que vimos antes ("dice 85%, gana el 76%") salía de un tramo con pocas partidas y del Elo de producción. Es un caso clásico de **muestra chica**: el patrón no se repite con otros datos.

In [7]:
def predict(name, rows):
    if name == "Base":
        return [r["p_elo"] for r in rows]
    if name == "A (todas)":
        return [scale_probability(r["p_elo"], T_all) for r in rows]
    if name == "A (cruces)":
        return [scale_probability(r["p_elo"], T_cross) for r in rows]
    scaler, m, cols = logistic[name]
    return list(m.predict_proba(scaler.transform(matrix(rows, cols)))[:, 1])

def worlds_series(rows, probs):
    """Series de Worlds: probabilidad del primer game llevada a la serie. El formato sale de las
    victorias del ganador (1 = Bo1, 2 = Bo3, 3 = Bo5)."""
    first, wins = {}, {}
    for r, p in zip(rows, probs):
        if r["page"] != "WLDs":
            continue
        first.setdefault(r["match_id"], (r["blue"], p))
        winner = r["blue"] if r["blue_won"] else r["red"]
        wins.setdefault(r["match_id"], {}).setdefault(winner, 0)
        wins[r["match_id"]][winner] += 1
    ps, os = [], []
    for mid, (team, p) in first.items():
        winner, n = max(wins[mid].items(), key=lambda kv: kv[1])
        ps.append(series_probability(p, 2 * n - 1))
        os.append(1.0 if winner == team else 0.0)
    return ps, os

def evaluate(names, predictor=predict):
    out = []
    for year in (2025, 2026):
        for label, subset in (("todas", data[year]), ("entre regiones", cross_region_records(data[year], regions[year]))):
            ys = [r["blue_won"] for r in subset]
            clusters = [r["match_id"] for r in subset]
            base = predict("Base", subset)
            for name in names:
                p = predictor(name, subset)
                ci = bootstrap_brier_diff(p, base, ys, clusters, reps=2000)
                out.append({"año": year, "partidas": label, "n": len(subset), "variante": name,
                            "brier": round(brier_score(p, ys), 4), "log_loss": round(log_loss(p, ys), 4),
                            "precisión": round(accuracy(p, ys), 3),
                            "Δ brier vs base": ci["diff"], "IC 95%": f"[{ci['low']:+.4f}, {ci['high']:+.4f}]"})
    return pd.DataFrame(out)

VARIANTS = ["Base", "A (todas)", "A (cruces)", "B0", "B-estilo", "B-roles", "B"]
results = evaluate(VARIANTS)
results

,año,partidas,n,variante,brier,log_loss,precisión,Δ brier vs base,IC 95%
0,2025,todas,3496,Base,0.2245,0.6403,0.654,0.000000,"[+0.0000, +0.0000]"
1,2025,todas,3496,A (todas),0.2210,0.6356,0.654,-0.003440,"[-0.0056, -0.0013]"
2,2025,todas,3496,A (cruces),0.2230,0.6373,0.654,-0.001436,"[-0.0019, -0.0010]"
3,2025,todas,3496,B0,0.2204,0.6319,0.646,-0.004042,"[-0.0054, -0.0027]"
4,2025,todas,3496,B-estilo,0.2193,0.6290,0.651,-0.005156,"[-0.0069, -0.0034]"
5,2025,todas,3496,B-roles,0.2172,0.6240,0.652,-0.007318,"[-0.0093, -0.0053]"
6,2025,todas,3496,B,0.2174,0.6244,0.647,-0.007047,"[-0.0092, -0.0049]"
7,2025,entre regiones,234,Base,0.2227,0.6352,0.632,0.000000,"[+0.0000, +0.0000]"
8,2025,entre regiones,234,A (todas),0.2465,0.7240,0.632,0.023771,"[+0.0120, +0.0366]"
9,2025,entre regiones,234,A (cruces),0.2261,0.6446,0.632,0.003372,"[+0.0008, +0.0061]"


## 5. Cómo leer la tabla

- **Δ brier vs base** negativo = mejor que el Elo actual.
- El **IC 95%** sale de un bootstrap por series:
  - se sortean series enteras con reposición 2000 veces y se mira cuánto varía la diferencia;
  - se sortean series y no partidas porque las partidas de una misma serie no son independientes.

  Si el intervalo no cruza el 0, es difícil que la diferencia sea suerte.
- En las partidas **entre regiones** hay solo ~150–230. Los intervalos son anchos y hay que ser humilde.

In [8]:
coefs = []
for name in ("B-estilo", "B-roles", "B"):
    scaler, m, cols = logistic[name]
    coefs.append(pd.Series(m.coef_[0], index=cols, name=name))
pd.concat(coefs, axis=1).round(3)

,B-estilo,B-roles,B
elo_logit,0.662,0.601,0.567
d_aggr,0.005,NaN,-0.027
d_dur,-0.074,NaN,-0.073
d_obj,0.162,NaN,0.108
x_aggr_dur,-0.003,NaN,0.001
x_obj_dur,-0.047,NaN,-0.043
d_top,NaN,0.186,0.176
d_jng,NaN,0.069,0.060
d_mid,NaN,0.078,0.058
d_bot,NaN,0.151,0.140


**Pesos estandarizados:** cuánto cambia el log-odds por cada desvío estándar de la variable.

- Las **variables de rol** pesan entre 0.06 y 0.19. El **top** y el **ADC** son los que más pesan.
- Las de **estilo** pesan casi 0. La única con algo de peso es la prioridad de objetivos, que es más "nivel" que "estilo".
- Las **interacciones** son prácticamente 0. Con estas medidas no aparece el efecto "piedra, papel o tijera".
  - Ojo: **ausencia de evidencia no es evidencia de ausencia**. Puede existir y que estas aproximaciones (kills al 15, duración) no lo capturen.
  - Para medirlo de verdad harían falta datos de draft y de composición.

**El problema de B0:** re-pesar el Elo con todas las partidas lo **empeora** entre regiones. La regresión aprende cuánto creerle al Elo mirando sobre todo partidas de liga (son el 93% de los datos), y esa confianza no es la adecuada para los cruces internacionales. El daño en los cruces no viene de las variables nuevas: viene de re-pesar el Elo.

## 6. Variante post-hoc: el Elo con peso fijo + forma por rol

Lo lógico, viendo lo anterior, es dejar el Elo **tal cual** (peso 1, sin re-pesarlo) y aprender solo cuánto agregan los roles encima. Técnicamente, el Elo entra como *offset*.

⚠️ **Esta variante se diseñó después de ver los resultados de 2025.** Eso la hace sospechosa: es justo el error de los caminos que se bifurcan. Por eso su evidencia fuerte es **2026**, que no se usó para diseñarla. Igual, ya habíamos mirado 2026 para las otras variantes, así que tampoco es una confirmación 100% limpia.

In [9]:
def fit_offset(rows, cols, lam=0.001):
    """Regresión logística con el log-odds del Elo fijo (peso 1) + penalización L2 sobre las variables nuevas."""
    X = matrix(rows, cols)
    mu, sd = X.mean(0), X.std(0)
    Z = (X - mu) / sd
    offset = np.array([r["elo_logit"] for r in rows])
    y = outcomes(rows)

    def loss(w):
        z = offset + w[0] + Z @ w[1:]
        return np.mean(np.logaddexp(0, z) - y * z) + lam * np.sum(w[1:] ** 2)

    w = minimize(loss, np.zeros(len(cols) + 1), method="BFGS").x

    def predictor(rows_):
        z = np.array([r["elo_logit"] for r in rows_]) + w[0] + ((matrix(rows_, cols) - mu) / sd) @ w[1:]
        return list(1 / (1 + np.exp(-z)))
    return predictor, pd.Series(w, index=("lado azul",) + cols).round(3)

roles_fixed, roles_fixed_w = fit_offset(train, ROLE_FEATURES)
print(roles_fixed_w.to_string())

def predict_plus(name, rows):
    return roles_fixed(rows) if name == "Elo + roles (post-hoc)" else predict(name, rows)

evaluate(["Base", "Elo + roles (post-hoc)"], predict_plus).query("variante != 'Base'")

lado azul    0.135
d_top        0.235
d_jng        0.090
d_mid        0.105
d_bot        0.192
d_sup        0.169


,año,partidas,n,variante,brier,log_loss,precisión,Δ brier vs base,IC 95%
1,2025,todas,3496,Elo + roles (post-hoc),0.2171,0.6237,0.648,-0.007375,"[-0.0097, -0.0051]"
3,2025,entre regiones,234,Elo + roles (post-hoc),0.2244,0.6435,0.645,0.001648,"[-0.0051, +0.0086]"
5,2026,todas,3367,Elo + roles (post-hoc),0.2225,0.6361,0.637,-0.005526,"[-0.0083, -0.0027]"
7,2026,entre regiones,152,Elo + roles (post-hoc),0.1808,0.5374,0.730,-0.015445,"[-0.0290, -0.0026]"


In [10]:
rows = []
for name in ["Base", "A (cruces)", "B0", "B", "Elo + roles (post-hoc)"]:
    ps, os = worlds_series(data[2025], predict_plus(name, data[2025]))
    rows.append({"variante": name, "series Worlds 2025": len(ps), "brier": round(brier_score(ps, os), 4),
                 "precisión": round(accuracy(ps, os), 3)})
pd.DataFrame(rows)

,variante,series Worlds 2025,brier,precisión
0,Base,44,0.1856,0.727
1,A (cruces),44,0.1861,0.727
2,B0,44,0.1858,0.750
3,B,44,0.1878,0.773
4,Elo + roles (post-hoc),44,0.1907,0.773


Worlds 2025 tiene **44 series**: demasiado pocas para separar modelos con Brier.
- B y "Elo + roles" aciertan 34 de 44 contra 32 de 44 del Elo, pero tienen un Brier un poco peor.
- Las dos diferencias son ruido. No sirven para decidir en ningún sentido.

In [11]:
def calibration_table(name, rows):
    """Calibración por tramos de la probabilidad del favorito, con 80%+ separado."""
    p = np.array(predict_plus(name, rows))
    y = outcomes(rows)
    fav_p = np.where(p >= 0.5, p, 1 - p)
    fav_won = np.where(p >= 0.5, y, 1 - y)
    bins = pd.cut(fav_p, [0.5, 0.6, 0.7, 0.8, 1.0], right=False, include_lowest=True)
    t = pd.DataFrame({"tramo": bins, "predicho": fav_p, "real": fav_won}).groupby("tramo", observed=True)
    return t.agg(n=("real", "size"), predicho=("predicho", "mean"), real=("real", "mean")).round(3)

cross_2025 = cross_region_records(data[2025], regions[2025])
pd.concat({name: calibration_table(name, cross_2025) for name in ["Base", "Elo + roles (post-hoc)"]}, axis=1)

Base                 Elo + roles (post-hoc)                
              n predicho   real                      n predicho   real
tramo                                                                 
[0.5, 0.6)   67    0.547  0.493                     60    0.548  0.517
[0.6, 0.7)   39    0.657  0.564                     44    0.650  0.568
[0.7, 0.8)   87    0.753  0.690                     68    0.754  0.706
[0.8, 1.0)   41    0.846  0.805                     62    0.853  0.758

In [12]:
print(f"Tiempo total: {time.time() - t0:.0f} s")

Tiempo total: 30 s


## 7. Conclusión

**A. Calibración por temperatura: no mejora. Se descarta.**
- Ajustada con todas las partidas, mejora dentro de las ligas pero **empeora** entre regiones (el IC no incluye el 0). No pasa la regla.
- Ajustada solo con los cruces de 2024 (T = 1.15), empeora un poco los cruces de 2025 y en 2026 no cambia nada.
- La "sobreconfianza" que habíamos visto era ruido de una muestra chica.

**B. Variables nuevas: sí mejora, pero solo por la forma por rol.**
- Con todas las partidas de 2025, B baja el Brier ~0.007 y el IC no incluye el 0. Pasa la condición 1.
- Entre regiones queda apenas peor que el Elo (+0.003), con el IC cruzando el 0. Técnicamente pasa la condición 2: no empeora de forma detectable.
- 2026 lo confirma con todas las partidas.
- El aporte viene **todo** de los roles. Estilo e interacciones no suman nada medible.
- Lo que empeora los cruces es re-pesar el Elo, no las variables nuevas.

**Candidata (post-hoc): Elo con peso fijo + forma por rol.**
- Mejora 2025 y 2026 con todas las partidas.
- Entre regiones es neutral en 2025 y mejora en 2026: −0.015, con el IC entero por debajo de 0.
- Es la mejor opción, con la advertencia de que se diseñó mirando los resultados.

**Limitaciones:**
- La LPL 2024/25 no tiene stats al minuto 15.
- Las partidas entre regiones son pocas.
- En el backtest se usan los jugadores que realmente jugaron. En producción hay que suponer la alineación antes del partido; casi siempre es la titular, pero un suplente sorpresa la cambia.
- Pasarlo al sitio implica bajar Oracle's Elixir (~80 MB, se actualiza a diario) en el pipeline.